# What Is a Qubit, in Words

Intuition first. Math after.

**Objectives:**
- Explain a qubit in plain English using the spinning-coin metaphor
- Map that metaphor onto a unit vector in $\mathbb{C}^2$
- Compute measurement probabilities of `|0>`, `|1>`, `|+>`, `|->` by hand
- Be ready to read `|psi> = alpha|0> + beta|1>` without flinching

**Reference:** See [`../GUIDE.md`](../GUIDE.md).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)

## 1. The spinning-coin metaphor

- A **classical bit** is a coin lying flat on the table. Heads or tails. That's it.
- A **qubit** is a coin you have just **spun**. While it spins, it is leaning   in some direction. The instant you stop the spin (measure), it falls to   heads or tails — but *which* one is probabilistic, and the probability   depends on the direction it was leaning.

So a qubit has more structure than a bit: it carries a *direction*, not just a value. The direction is what makes superposition and interference possible. The value (0 or 1) only appears at measurement.

The rest of this notebook makes "direction" precise.

## 2. The math version: a unit 2-vector

A qubit state is a 2-vector

$$|\psi\rangle = \begin{pmatrix} \alpha \\ \beta \end{pmatrix}$$

with complex entries `alpha`, `beta` satisfying $|\alpha|^2 + |\beta|^2 = 1$.

The two **computational basis states** are

$$|0\rangle = \begin{pmatrix} 1 \\ 0 \end{pmatrix}, \qquad |1\rangle = \begin{pmatrix} 0 \\ 1 \end{pmatrix}.$$

Every other state is some combination of those two.

In [ ]:
# Build the four canonical states.
zero  = np.array([1, 0], dtype=complex)
one   = np.array([0, 1], dtype=complex)
plus  = (zero + one) / np.sqrt(2)        # |+> = (|0> + |1>) / sqrt(2)
minus = (zero - one) / np.sqrt(2)        # |-> = (|0> - |1>) / sqrt(2)

for name, state in [('|0>', zero), ('|1>', one), ('|+>', plus), ('|->', minus)]:
    print(f'{name}: {state}    norm={np.linalg.norm(state):.3f}')

## 3. Measurement, formally

If $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ then measuring in the computational basis gives outcome **0** with probability $|\alpha|^2$ and outcome **1** with probability $|\beta|^2$.

That is the **Born rule**, and it is the only mysterious thing in quantum mechanics. Once you accept it, the rest is linear algebra.

In [ ]:
def measurement_probs(psi):
    return np.abs(psi) ** 2

for name, state in [('|0>', zero), ('|1>', one), ('|+>', plus), ('|->', minus)]:
    p = measurement_probs(state)
    print(f'{name}: P(0) = {p[0]:.3f},  P(1) = {p[1]:.3f}')

**Read this slowly:** `|+>` and `|->` give *identical* measurement probabilities (50/50). They are different states — but you cannot tell them apart with a single computational-basis measurement. To distinguish them you have to *do* something to the state first (e.g. apply a Hadamard) and then measure. That is what gates are for.

## 4. Simulate measurement

Combining notebook 3 with what we just learned: "running a quantum circuit with N shots" is *literally* sampling N times from the distribution `|psi|^2`.

In [ ]:
def simulate_measurements(psi, n_shots=1000, seed=0):
    rng = np.random.default_rng(seed)
    probs = measurement_probs(psi)
    return rng.choice(len(psi), size=n_shots, p=probs)

shots = simulate_measurements(plus, n_shots=2000)
p0 = np.mean(shots == 0); p1 = np.mean(shots == 1)
print(f'|+> measured 2000 times: P(0) ~ {p0:.3f}, P(1) ~ {p1:.3f}')

In [ ]:
# Plot all four canonical states for comparison.
states = [('|0>', zero), ('|1>', one), ('|+>', plus), ('|->', minus)]
fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharey=True)
for ax, (name, psi) in zip(axes, states):
    p = measurement_probs(psi)
    ax.bar(['0', '1'], p, color=['#ff9900', '#146eb4'])
    ax.set_title(name)
    ax.set_ylim(0, 1)
axes[0].set_ylabel('Probability')
plt.suptitle('Measurement distributions of the four canonical states')
plt.tight_layout()
plt.show()

## 5. A non-canonical state

Let's build a custom state and watch the probabilities behave.

In [ ]:
# 30% chance of 0, 70% chance of 1
psi = np.array([np.sqrt(0.3), np.sqrt(0.7)], dtype=complex)
print('state:', psi)
print('norm:', np.linalg.norm(psi))           # 1
print('measurement probs:', measurement_probs(psi))

shots = simulate_measurements(psi, n_shots=10_000, seed=1)
print(f'empirical: P(0) ~ {np.mean(shots == 0):.3f}, P(1) ~ {np.mean(shots == 1):.3f}')

## 6. Self-check

In [ ]:
# Q1: Build the state psi = (1/2)|0> + (sqrt(3)/2)|1>.
# What are P(0) and P(1)?
# TODO


In [ ]:
# Q2: A qubit is prepared in the state psi = [1/sqrt(2), 1j/sqrt(2)].
# What are the measurement probabilities? (Note the imaginary amplitude.)
# TODO


In [ ]:
# Q3: Why do |+> and |-> give identical measurement probabilities, even though
# they are different states? Write one sentence in the cell below.


*Answer (Q3): Because the Born rule depends on $|\alpha|^2$ and $|\beta|^2$ only, the **sign** of $\beta$ does not show up in a computational-basis measurement. To detect the sign difference you have to apply a gate (specifically, a Hadamard) before measuring.*

### Solutions

In [ ]:
# --- Q1 ---
psi = np.array([1/2, np.sqrt(3)/2], dtype=complex)
print(measurement_probs(psi))    # [0.25, 0.75]

# --- Q2 ---
psi = np.array([1, 1j]) / np.sqrt(2)
print(measurement_probs(psi))    # [0.5, 0.5] — imaginary part doesn't change |.|^2

**You finished notebook 4.** Move on to [`05-dirac-notation-decoded.ipynb`](05-dirac-notation-decoded.ipynb).